# Bloco 3 — Notebook 02: Simulador Causal Single-Slot, Ablação (Degraus A–E) e DSR
- Partição rígida: **60% Treino / 25% Validação / 15% Holdout (6 meses)** com *Purging* de 24h e *Embargo* de 2%.
- Cálculo empírico de $\mathbb{E}[R]$ (incluindo o desfecho real das saídas por barreira de tempo de 24h).
- Ablação explícita do Degrau C com e sem filtro de *Order Flow Delta* (`require_order_flow_filter`).
- Resolução de 1 minuto (`1m`) para testar o impacto do Breakeven em $+1,2R$ (Degrau E).
- Registro automático em `logs/backtest_trials_registry.csv` ($K_{\max} = 20$) e validação do **Gate 3** ($\text{DSR} \ge 0,95$).


In [ ]:
# Se estiver rodando no Google Colab, clona o repositório e instala dependências leves
import os, sys
from pathlib import Path
if 'google.colab' in sys.modules and not Path('config/strategies.yaml').exists():
    !git clone https://github.com/pac-coins/quant-crypto-chameleon-radar-engine.git
    os.chdir('quant-crypto-chameleon-radar-engine')
    !pip install -q pydantic pydantic-settings PyYAML pandas requests cryptography
if '.' not in sys.path:
    sys.path.insert(0, '.')


In [ ]:
from pathlib import Path
from src.core.config import load_config
from src.regimes.regime_detector import RegimeDetector
from src.strategies.bull_trend_pullback import BullTrendPullbackStrategy
from src.backtest.causal_simulator import CausalSingleSlotSimulator, CostScenario
from src.backtest.ablation_and_dsr import (
    split_temporal_vaults,
    compute_afml_sample_uniqueness,
    compute_deflated_sharpe_ratio,
    register_backtest_trial,
    evaluate_gate3_criteria,
)

cfg = load_config()
detector = RegimeDetector(cfg.strategies.regimes_and_strategies.regime_thresholds_4h)
bull_strategy = BullTrendPullbackStrategy(enabled=True)
REGISTRY_CSV = Path('logs/backtest_trials_registry.csv')


In [ ]:
def run_ablation_and_gate3(aligned_frames, intracandle_1m_map=None):
    tcol = 'close_time_ms' if 'close_time_ms' in next(iter(aligned_frames.values())).columns else 'close_time'
    all_ts = sorted({int(t) for df in aligned_frames.values() for t in df[tcol].tolist()})
    bounds = split_temporal_vaults(all_ts, train_pct=0.60, val_pct=0.25)

    sim_base = CausalSingleSlotSimulator(cost_scenario=CostScenario.BASE)
    sim_pess = CausalSingleSlotSimulator(cost_scenario=CostScenario.PESSIMISTIC)

    # Degrau B: Sem filtro de clima 4h
    trades_step_b = sim_base.run_multi_asset_simulation(
        aligned_frames, [bull_strategy], detector, ignore_regime_filter=True,
        start_time_ms=bounds.val_start_ms, end_time_ms=bounds.val_end_ms
    )

    # Degrau C (Puro vs Com Filtro de Order Flow Delta)
    trades_step_c_pure = sim_base.run_multi_asset_simulation(
        aligned_frames, [bull_strategy], detector, require_order_flow_filter=False,
        start_time_ms=bounds.val_start_ms, end_time_ms=bounds.val_end_ms
    )
    trades_step_c_pess = sim_pess.run_multi_asset_simulation(
        aligned_frames, [bull_strategy], detector, require_order_flow_filter=False,
        start_time_ms=bounds.val_start_ms, end_time_ms=bounds.val_end_ms
    )
    trades_step_c_of = sim_base.run_multi_asset_simulation(
        aligned_frames, [bull_strategy], detector, require_order_flow_filter=True,
        start_time_ms=bounds.val_start_ms, end_time_ms=bounds.val_end_ms
    )

    # Degrau E: Breakeven em +1.2R resolvido com velas de 1 minuto (1m)
    sim_be = CausalSingleSlotSimulator(cost_scenario=CostScenario.BASE, breakeven_enabled=True, breakeven_trigger_r=1.2)
    trades_step_e = sim_be.run_multi_asset_simulation(
        aligned_frames, [bull_strategy], detector, intracandle_1m_map=intracandle_1m_map,
        start_time_ms=bounds.val_start_ms, end_time_ms=bounds.val_end_ms
    )

    gate3_report = evaluate_gate3_criteria(
        trades_base=trades_step_c_pure,
        trades_pessimistic=trades_step_c_pess,
        holdout_start_ms=bounds.holdout_start_ms,
        n_trials=2,
    )
    return {
        'bounds': bounds,
        'step_b': trades_step_b,
        'step_c_pure': trades_step_c_pure,
        'step_c_order_flow': trades_step_c_of,
        'step_e_breakeven_1m': trades_step_e,
        'gate3_report': gate3_report,
    }
